In [1]:
# -*- coding: utf-8 -*-
"""

    create_flow_data.py

        Create a nodal flow file from spatial data

"""

#=======================
# Modules

import geopandas as gpd

# Add local directory to path
import sys
sys.path.append("../../")

# Import local functions
from utils import *

#=======================
# PROCESSING

def compute_supply_demand(nodes):
    '''Calculate demand based on population and ei
    '''
    nodes['flow'] = 0
    # supply
    # capacity (MW) * 10 ** 3 = kW
    nodes.loc[nodes.asset_type == 'source', 'flow'] = \
        nodes.loc[nodes.asset_type == 'source', 'capacity'] * 10 ** 6
    # demand
    # persons * kW/person = kW
    nodes.loc[nodes.asset_type == 'sink', 'flow'] = \
        nodes.loc[nodes.asset_type == 'sink', 'population'] * \
            nodes.loc[nodes.asset_type == 'sink', 'ei'] 
    return nodes

# path_to_nodes = '../data/demo/nodes_demo.shp'
# path_to_edges = '../data/demo/edges_demo.shp'

path_to_nodes = '../data/spatial/infrasim-network/nodes.shp'
path_to_edges = '../data/spatial/infrasim-network/edges.shp'

# read data
network = read_data(path_to_nodes=path_to_nodes,
                    path_to_edges=path_to_edges)

print('loaded data')

# flow nodes
flow_nodes = get_flow_nodes(network).reset_index(drop=True)
flow_nodes = compute_supply_demand(flow_nodes)
flow_nodes['flow'] = flow_nodes['flow'] 

# pivot
flow_nodes = flow_nodes[['id','flow']]
list_of_nodes = flow_nodes.id.to_list()
flow_nodes = flow_nodes.pivot_table(columns='id').reset_index(drop=True)
flow_nodes['timestep'] = 1
flow_nodes = flow_nodes[['timestep'] + list_of_nodes]
flow_nodes = flow_nodes.astype('int64')

# save file
flow_nodes.to_csv('../data/csv/generated_nodal_flows.csv',index=False)

print('done')

/Users/amanmajid/opt/anaconda3/envs/jem_model/lib/python3.7/site-packages/geopandas/_compat.py:58: UserWarning: The installed version of PyGEOS is too old (0.6 installed, 0.8 required), and thus GeoPandas will not use PyGEOS.
  UserWarning,


loaded data
done


In [10]:
flow_nodes = get_flow_nodes(network).reset_index(drop=True)
flow_nodes = compute_supply_demand(flow_nodes)
flow_nodes['flow'] = flow_nodes['flow'] 
flow_nodes = flow_nodes[['id','flow']]
list_of_nodes = flow_nodes.id.to_list()
flow_nodes = flow_nodes.pivot_table(columns='id').reset_index(drop=True)
flow_nodes['timestep'] = 1
flow_nodes = flow_nodes[['timestep'] + list_of_nodes]
flow_nodes = flow_nodes.astype('int64')

flow_nodes

id,timestep,node_1,node_67,node_80,node_84,node_85,node_87,node_89,node_91,node_95,...,node_42949,node_42950,node_42951,node_42952,node_42954,node_42956,node_42958,node_42959,node_42966,node_42967
0,1,20000000,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
flow_nodes['node_11']

0    40000000
Name: node_11, dtype: int64

In [ ]:
flow_nodes[flow_nodes.asset_type == 'source']